## RAG Assignment: Baseline vs. Re-Ranking

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0" 

import re, time, json, collections
import numpy as np
import fitz # PyMuPDF
import faiss
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

/home/priyanka/miniconda3/envs/myenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
EMB_MODEL     = "BAAI/bge-m3"
LLM_NAME      = "Qwen/Qwen3-8B"
CHUNK_SIZE    = 1000  # characters (~250 tokens)
CHUNK_OVERLAP = 150
TOP_K         = 10 # retrieved candidates
TOP_N         = 3 # chunks given to the LLM
MAX_NEW_TOK   = 300
DEVICE        = "cuda"

### 1. Dataset


In [3]:
# Appel's annual report

PDF_PATH = "data/report.pdf" 

In [4]:
# extract text page by page
doc = fitz.open(PDF_PATH)
pages = []
for i, page in enumerate(doc):
    # sort=True gives a more natural reading order for multi-column layouts
    pages.append({"page": i + 1, "text": page.get_text("text", sort=True)})
print("pages:", len(pages))

pages: 80


In [5]:
def find_boilerplate(pages, min_frac=0.3):
    """Lines that appear on >= min_frac of pages are treated as headers/footers."""
    counter = collections.Counter()
    for p in pages:
        # digits -> "#" so page-numbered footers ("... | 10-K | 35") match across pages
        counter.update({re.sub(r"\d+", "#", ln.strip()) for ln in p["text"].splitlines() if ln.strip()})
    thresh = max(3, int(len(pages) * min_frac))
    return {ln for ln, c in counter.items() if c >= thresh and len(ln) < 120}
 
boiler = find_boilerplate(pages)
print("boilerplate lines removed:", list(boiler))

boilerplate lines removed: ['Apple Inc. | # Form #-K | #']


In [6]:
def clean(text):
    lines = [ln for ln in text.splitlines() if re.sub(r"\d+", "#", ln.strip()) not in boiler]
    text = "\n".join(lines)
    text = re.sub(r"-\n(\w)", r"\1", text)      # join hyphenated line breaks
    text = re.sub(r"\n{2,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)  # single newlines -> space
    return text.strip()

In [7]:
for p in pages:
    p["text"] = clean(p["text"])

### 2. Two-Stage Retrieval

#### A. Baseline Retrieval - Pipeline 1

In [8]:
# chunking (within each page, so page metadata stays exact)
def chunk_text(text, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    chunks, start, n = [], 0, len(text)
    while start < n:
        end = min(start + size, n)
        if end < n:                                   # snap to a sentence / space boundary
            cut = text.rfind(". ", start, end)
            if cut > start + size * 0.6:
                end = cut + 1
            else:
                sp = text.rfind(" ", start, end)
                if sp > start + size * 0.6:
                    end = sp
        piece = text[start:end].strip()
        if len(piece) > 50:
            chunks.append(piece)
        if end >= n:
            break
        start = max(end - overlap, start + 1)
        sp = text.find(" ", start, end)              # don't start mid-word
        if sp != -1:
            start = sp + 1
    return chunks

In [9]:
chunks = []   # list of dicts: chunk_id, page, text
for p in pages:
    for t in chunk_text(p["text"]):
        chunks.append({"chunk_id": len(chunks), "page": p["page"], "text": t})
print("total chunks:", len(chunks))
 
import random
for c in random.sample(chunks, 5):
    print(f"--- chunk {c['chunk_id']} (page {c['page']}) ---\n{c['text']}\n")
 
# helpers for finding gold chunks when you write failure queries later
def show_chunk(i):
    c = chunks[i]
    print(f"[chunk {c['chunk_id']} | page {c['page']}]\n{c['text']}")
 
def find_chunks(keyword, limit=20):
    kw = keyword.lower()
    hits = [c["chunk_id"] for c in chunks if kw in c["text"].lower()]
    print(f"'{keyword}' appears in {len(hits)} chunks:", hits[:limit])
    return hits

total chunks: 376
--- chunk 144 (page 24) ---
Item 7. Management’s Discussion and Analysis of Financial Condition and Results of Operations

The following discussion should be read in conjunction with the consolidated financial statements and accompanying notes included in Part II, Item 8 of this Form 10-K. This Item generally discusses 2025 and 2024 items and year-to-year comparisons between 2025 and 2024. Discussions of 2023 items and year-to-year comparisons between 2024 and 2023 are not included, and can be found in “Management’s Discussion and Analysis of Financial Condition and Results of Operations” in Part II, Item 7 of the Company’s Annual Report on Form 10-K for the fiscal year ended September 28, 2024.

Product, Service and Software Announcements

The Company announces new product, service and software offerings at various times during the year.

--- chunk 7 (page 3) ---
Apple Inc.

 Form 10-K

 For the Fiscal Year Ended September 27, 2025

 TABLE OF CONTENTS

 Page

 Part I

In [10]:
# embed chunks with the bi-encoder and build the FAISS index
embedder = SentenceTransformer(EMB_MODEL, device=DEVICE)
embedder.half()
embedder.max_seq_length = 512
 
texts = [c["text"] for c in chunks]
emb_path = "chunk_embeddings.npy"
if os.path.exists(emb_path):
    chunk_emb = np.load(emb_path)
else:
    chunk_emb = embedder.encode(texts, batch_size=64, normalize_embeddings=True,
                                convert_to_numpy=True, show_progress_bar=True)
    np.save(emb_path, chunk_emb)
chunk_emb = np.ascontiguousarray(chunk_emb.astype("float32"))
print("embeddings:", chunk_emb.shape)
 
# normalized vectors => inner product == cosine similarity
index = faiss.IndexFlatIP(chunk_emb.shape[1])
index.add(chunk_emb)
print("vectors in index:", index.ntotal)

Batches: 100%|██████████| 6/6 [00:00<00:00,  7.26it/s]

embeddings: (376, 1024)
vectors in index: 376


In [11]:
# retrieval function (top 10) + NumPy cross-check
def embed_query(q):
    return embedder.encode([q], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
 
def retrieve_topk(query, k=TOP_K):
    qv = embed_query(query)
    scores, ids = index.search(qv, k)
    return [(int(i), float(s)) for i, s in zip(ids[0], scores[0])]

In [12]:
# verify FAISS against a plain NumPy dot product (should give the same ranking)
q = "test query"
qv = embed_query(q)
np_top = np.argsort(-(chunk_emb @ qv[0]))[:TOP_K].tolist()
assert np_top == [i for i, _ in retrieve_topk(q)], "FAISS and NumPy rankings differ!"
print("FAISS == NumPy check passed")

FAISS == NumPy check passed


In [13]:
# load Qwen3-8B
tok = AutoTokenizer.from_pretrained(LLM_NAME)
llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, torch_dtype=torch.bfloat16, device_map=DEVICE)
llm.eval()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 399/399 [00:04<00:00, 85.37it/s]


Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 4096)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=4096, out_features=12288, bias=False)
          (up_proj): Linear(in_features=4096, out_features=12288, bias=False)
          (down_proj): Linear(in_features=12288, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((4096,), eps=1e-06)
        (post_attention_la

In [14]:
# %% Cell 8: ONE prompt builder + ONE generate function, reused by BOTH pipelines
SYSTEM_PROMPT = (
    "You are a careful assistant answering questions about a document. "
    "Use ONLY the provided context. If the context does not contain the answer, "
    "say 'The context does not contain this information.' Be concise and precise, "
    "and mention any conditions, exceptions or restrictions stated in the context."
)
 
def build_prompt(query, chunk_ids):
    ctx = "\n\n".join(
        f"[Chunk {i} | page {chunks[i]['page']}]\n{chunks[i]['text']}" for i in chunk_ids
    )
    user = f"Context:\n{ctx}\n\nQuestion: {query}\n\nAnswer:"
    return [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user}]
 
@torch.no_grad()
def generate(messages, max_new_tokens=MAX_NEW_TOK):
    # enable_thinking=False turns off Qwen3's <think> mode so answers are short and comparable
    text = tok.apply_chat_template(messages, tokenize=False,
                                   add_generation_prompt=True, enable_thinking=False)
    inputs = tok(text, return_tensors="pt").to(llm.device)
    out = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                       temperature=None, top_p=None, top_k=None)   # greedy: deterministic
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
 

In [15]:
# Pipeline 1 end to end
def sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()
 
def baseline(query):
    t0 = time.perf_counter()
    top10 = retrieve_topk(query, TOP_K)          # [(chunk_id, cosine), ...] sorted desc
    sync(); t_retr = time.perf_counter() - t0
 
    top3_ids = [i for i, _ in top10[:TOP_N]]     # take the top 3 directly
    t1 = time.perf_counter()
    answer = generate(build_prompt(query, top3_ids))
    sync(); t_llm = time.perf_counter() - t1
 
    return {"query": query, "top10": top10, "top3_ids": top3_ids, "answer1": answer,
            "retrieval_ms": t_retr * 1000, "llm_s": t_llm}

In [16]:
def print_result(r):
    print("QUERY:", r["query"])
    print("Top-10 (rank, chunk_id, page, cosine):")
    for rank, (i, s) in enumerate(r["top10"], 1):
        mark = " <- sent to LLM" if rank <= TOP_N else ""
        print(f"  #{rank:<2} chunk {i:<5} p.{chunks[i]['page']:<4} {s:.4f}{mark}")
    print("ANSWER 1:", r["answer1"])
    print(f"retrieval {r['retrieval_ms']:.1f} ms | LLM {r['llm_s']:.1f} s\n")
 
# %% Cell 10: smoke test (replace with a real question about YOUR document)
res = baseline("what is the report about?")
print_result(res)

QUERY: what is the report about?
Top-10 (rank, chunk_id, page, cosine):
  #1  chunk 263   p.55   0.5195 <- sent to LLM
  #2  chunk 254   p.54   0.5189 <- sent to LLM
  #3  chunk 9     p.4    0.5170 <- sent to LLM
  #4  chunk 372   p.79   0.4996
  #5  chunk 367   p.78   0.4978
  #6  chunk 267   p.57   0.4962
  #7  chunk 253   p.54   0.4952
  #8  chunk 32    p.7    0.4915
  #9  chunk 246   p.52   0.4910
  #10 chunk 371   p.79   0.4855
ANSWER 1: The report is about the Company's internal control over financial reporting, including the audit opinion provided by Ernst & Young LLP, the absence of changes in internal control during the fourth quarter of 2025, and the basis for the audit opinion. It also mentions that the report contains forward-looking statements regarding the Company's business and results of operations.
retrieval 15.7 ms | LLM 1.6 s



#### B. Re-Ranking - Pipeline 2

In [17]:
from sentence_transformers import CrossEncoder

CE_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"

# the cross-encoder reads (query, chunk) as ONE sequence -> joint attention over both
reranker = CrossEncoder(CE_MODEL, device=DEVICE, max_length=512)

# warm-up so the first timed query doesn't include CUDA / kernel start-up
reranker.predict([("warm up", chunks[0]["text"])])
embed_query("warm up")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 8514.95it/s]


array([[ 0.02116394,  0.02047729, -0.04849243, ..., -0.00387192,
        -0.01230621,  0.03897095]], shape=(1, 1024), dtype=float32)

In [18]:
def rerank(query, top10):
    """Score all 10 (query, chunk) pairs with the cross-encoder, sort descending.
    Returns [(chunk_id, ce_score, bi_rank), ...] where bi_rank is the original rank (1-10)."""
    pairs = [(query, chunks[i]["text"]) for i, _ in top10]
    ce_scores = reranker.predict(pairs, batch_size=len(pairs), convert_to_numpy=True)
    ranked = [(i, float(s), r) for r, ((i, _), s) in enumerate(zip(top10, ce_scores), 1)]
    return sorted(ranked, key=lambda x: x[1], reverse=True)

def reranked_pipeline(query):
    t0 = time.perf_counter()
    top10 = retrieve_topk(query, TOP_K)          # same top-10 candidates as Pipeline 1
    sync(); t_retr = time.perf_counter() - t0

    t1 = time.perf_counter()
    ranked = rerank(query, top10)
    sync(); t_rr = time.perf_counter() - t1

    top3_ids = [i for i, _, _ in ranked[:TOP_N]] # top 3 after re-ranking
    t2 = time.perf_counter()
    answer = generate(build_prompt(query, top3_ids))   # identical prompt structure
    sync(); t_llm = time.perf_counter() - t2

    return {"query": query, "top10": top10, "reranked": ranked, "top3_ids": top3_ids,
            "answer2": answer, "retrieval_ms": t_retr * 1000, "rerank_ms": t_rr * 1000,
            "llm_s": t_llm}

In [19]:
def compare(query, show_text=False):
    """Run both pipelines on one query and print rank shifts, answers and latency."""
    r1 = baseline(query)
    r2 = reranked_pipeline(query)
    assert [i for i, _ in r1["top10"]] == [i for i, _ in r2["top10"]]   # same candidates

    cos = dict(r1["top10"])
    print("QUERY:", query)
    print(f"{'CE rank':<8}{'chunk':<7}{'page':<6}{'bi rank':<9}{'cosine':<9}{'CE score':<10}shift")
    for new_rank, (i, ce, old_rank) in enumerate(r2["reranked"], 1):
        shift = old_rank - new_rank
        arrow = f"#{old_rank} -> #{new_rank}" + (f"  (+{shift})" if shift > 0 else f"  ({shift})" if shift < 0 else "")
        flag = []
        if new_rank <= TOP_N: flag.append("P2")
        if old_rank <= TOP_N: flag.append("P1")
        print(f"#{new_rank:<7}{i:<7}{chunks[i]['page']:<6}#{old_rank:<8}{cos[i]:<9.4f}{ce:<10.3f}{arrow}  {'/'.join(flag)}")

    promoted = [i for i in r2["top3_ids"] if i not in r1["top3_ids"]]
    dropped  = [i for i in r1["top3_ids"] if i not in r2["top3_ids"]]
    print(f"\nPipeline 1 top-3: {r1['top3_ids']}   Pipeline 2 top-3: {r2['top3_ids']}")
    print(f"promoted into top-3: {promoted}   dropped from top-3: {dropped}")
    if show_text:
        for i in promoted:
            print(); show_chunk(i)

    print("\nANSWER 1 (baseline):\n", r1["answer1"])
    print("\nANSWER 2 (re-ranked):\n", r2["answer2"])
    print(f"\nlatency  P1: retrieval {r1['retrieval_ms']:.1f} ms | LLM {r1['llm_s']:.2f} s")
    print(f"         P2: retrieval {r2['retrieval_ms']:.1f} ms + rerank {r2['rerank_ms']:.1f} ms | LLM {r2['llm_s']:.2f} s\n")
    return r1, r2

In [20]:
# smoke test (replace with your failure queries)
r1, r2 = compare("How much revenue did Apple make in China?", show_text=True)

QUERY: How much revenue did Apple make in China?
CE rank chunk  page  bi rank  cosine   CE score  shift
#1      197    39    #4       0.6008   5.684     #4 -> #1  (+3)  P2
#2      196    39    #1       0.6269   0.478     #1 -> #2  (-1)  P2/P1
#3      150    25    #2       0.6259   -0.118    #2 -> #3  (-1)  P2/P1
#4      175    32    #3       0.6080   -1.079    #3 -> #4  (-1)  P1
#5      183    36    #7       0.5678   -1.766    #7 -> #5  (+2)  
#6      177    33    #5       0.5979   -2.215    #5 -> #6  (-1)  
#7      242    50    #6       0.5702   -3.233    #6 -> #7  (-1)  
#8      243    50    #9       0.5453   -3.340    #9 -> #8  (+1)  
#9      244    51    #8       0.5455   -4.884    #8 -> #9  (-1)  
#10     151    26    #10      0.5408   -6.940    #10 -> #10  

Pipeline 1 top-3: [196, 150, 175]   Pipeline 2 top-3: [197, 196, 150]
promoted into top-3: [197]   dropped from top-3: [175]

[chunk 197 | page 39]
2025, 2024 and 2023, except in Greater China, where iPhone revenue represente

### 3. Failure Mode Analysis

In [21]:
# Each case: (category, query, gold phrase). The gold chunk is whichever chunk contains
# the gold phrase -> found by text, so it survives re-chunking. Ranks in the comments
# are what we observed (bi-encoder rank -> cross-encoder rank).
FAILURE_QUERIES = [
    # keyword trap: top-1 bi-encoder hit is "The Company designs, manufactures and markets..."
    ("Keyword trap", "Where are Apple's products manufactured?",
     "manufactured by outsourcing partners"),                                    # #4 -> #1
    # exception: general Additional Amounts covenant outranks the "shall not apply" carve-outs
    ("Exception", "Are there exceptions to Apple's obligation to pay Additional Amounts?",
     "shall not apply"),                                                         # #4 -> #1
    # constraint: tax redemption is allowed only "in whole, but not in part"
    ("Negation / constraint", "Can Apple redeem the Notes partially if tax laws change?",
     "in whole, but not in part"),                                               # #4 -> #1
    # negation: Additional Amounts are paid only to holders who are NOT US persons
    ("Negation / constraint", "Does Apple pay additional amounts to US holders?",
     "to a holder who is not a United States person"),                           # #7 -> #2
    # buried: bi-encoder prefers the State Aid history + financial statements, not the $ amount
    ("Buried chunk", "How much did Apple pay in taxes because of the EU State Aid ruling?",
     "one-time income tax charge of $10.2 billion"),                             # #5 -> #2
    ("Buried chunk", "Did Apple change its internal controls in the fourth quarter?",
     "There were no changes in the Company’s internal control"),                 # #8 -> #1
    # distractor: Q4-only monthly repurchase table (89,498K shares) vs full-year figure
    ("Distractor", "How many shares did Apple repurchase in fiscal 2025?",
     "repurchased 402 million shares"),                                          # #8 -> #2
]

# cases where re-ranking does NOT help (for the report's limitations discussion)
LIMITATION_QUERIES = [
    ("Recall failure", "What was the financial impact of the European Court of Justice decision on Apple?",
     "one-time income tax charge of $10.2 billion"),                             # not in top 10
    ("Recall failure", "Which executive is departing Apple?",
     "will transition from his role"),                                           # not in top 10
    ("Re-ranker demotes gold", "Which taxes are excluded from Additional Amounts?",
     "(3) to any Tax that would not have been imposed but for"),                 # #6 -> #7
]

def gold_ids(phrase):
    p = phrase.lower()
    return [c["chunk_id"] for c in chunks if p in re.sub(r"\s+", " ", c["text"]).lower()]

In [22]:
def run_case(category, query, phrase):
    gold = gold_ids(phrase)
    assert gold, f"gold phrase not found: {phrase!r}"
    print("=" * 100); print(f"[{category}]  gold chunk(s): {gold}")
    r1, r2 = compare(query)
    bi = {i: r for r, (i, _) in enumerate(r1["top10"], 1)}
    ce = {i: r for r, (i, _, _) in enumerate(r2["reranked"], 1)}
    g = min(gold, key=lambda i: bi.get(i, 99))            # best-ranked gold chunk
    return {"category": category, "query": query, "gold_chunk": g,
            "bi_rank": bi.get(g, ">10"), "ce_rank": ce.get(g, ">10"),
            "rank_shift": " -> ".join(f"#{x[g]}" if g in x else "not in top 10" for x in (bi, ce)),
            "gold_in_P1_top3": g in r1["top3_ids"], "gold_in_P2_top3": g in r2["top3_ids"],
            "answer1": r1["answer1"], "answer2": r2["answer2"],
            "p1_retrieval_ms": r1["retrieval_ms"], "p2_retrieval_ms": r2["retrieval_ms"],
            "rerank_ms": r2["rerank_ms"], "p1_llm_s": r1["llm_s"], "p2_llm_s": r2["llm_s"]}

results = [run_case(*case) for case in FAILURE_QUERIES]

[Keyword trap]  gold chunk(s): [238, 239]
QUERY: Where are Apple's products manufactured?
CE rank chunk  page  bi rank  cosine   CE score  shift
#1      239    49    #4       0.5657   3.225     #4 -> #1  (+3)  P2
#2      11     4     #1       0.6133   -0.276    #1 -> #2  (-1)  P2/P1
#3      360    76    #3       0.5678   -1.858    #3 -> #3  P2/P1
#4      57     11    #8       0.5275   -1.982    #8 -> #4  (+4)  
#5      12     4     #2       0.6004   -3.094    #2 -> #5  (-3)  P1
#6      15     5     #5       0.5453   -5.006    #5 -> #6  (-1)  
#7      175    32    #6       0.5419   -5.093    #6 -> #7  (-1)  
#8      145    24    #7       0.5318   -7.892    #7 -> #8  (-1)  
#9      186    37    #10      0.5220   -9.290    #10 -> #9  (+1)  
#10     196    39    #9       0.5231   -10.110   #9 -> #10  (-1)  

Pipeline 1 top-3: [11, 12, 360]   Pipeline 2 top-3: [239, 11, 360]
promoted into top-3: [239]   dropped from top-3: [12]

ANSWER 1 (baseline):
 The context does not contain this inform

In [23]:
import pandas as pd
pd.set_option("display.max_colwidth", 80)

df = pd.DataFrame(results)
df[["category", "query", "gold_chunk", "rank_shift", "gold_in_P1_top3", "gold_in_P2_top3"]]

,category,query,gold_chunk,rank_shift,gold_in_P1_top3,gold_in_P2_top3
0,Keyword trap,Where are Apple's products manufactured?,239,#4 -> #1,False,True
1,Exception,Are there exceptions to Apple's obligation to pay Additional Amounts?,310,#4 -> #1,False,True
2,Negation / constraint,Can Apple redeem the Notes partially if tax laws change?,318,#4 -> #1,False,True
3,Negation / constraint,Does Apple pay additional amounts to US holders?,309,#7 -> #2,False,True
4,Buried chunk,How much did Apple pay in taxes because of the EU State Aid ruling?,215,#5 -> #2,False,True
5,Buried chunk,Did Apple change its internal controls in the fourth quarter?,263,#8 -> #1,False,True
6,Distractor,How many shares did Apple repurchase in fiscal 2025?,230,#8 -> #2,False,True


In [24]:
# side-by-side answers for the report (Answer 1 vs Answer 2)
for r in results:
    print("=" * 100)
    print(f"[{r['category']}] {r['query']}   (gold chunk {r['gold_chunk']}: #{r['bi_rank']} -> #{r['ce_rank']})")
    print("\nANSWER 1:", r["answer1"])
    print("\nANSWER 2:", r["answer2"], "\n")

[Keyword trap] Where are Apple's products manufactured?   (gold chunk 239: #4 -> #1)

ANSWER 1: The context does not contain this information.

ANSWER 2: Apple's products are primarily manufactured by outsourcing partners located in China mainland, India, Japan, South Korea, Taiwan, and Vietnam. The context does not specify the exact locations of all manufacturing sites, only the primary regions. 

[Exception] Are there exceptions to Apple's obligation to pay Additional Amounts?   (gold chunk 310: #4 -> #1)

ANSWER 1: Yes, there are exceptions to Apple's obligation to pay Additional Amounts. The context indicates that the obligation to pay Additional Amounts is subject to "exceptions and limitations set forth below," although the specific exceptions and limitations are not detailed in the provided context.

ANSWER 2: Yes, there are exceptions to Apple's obligation to pay Additional Amounts. The obligation does not apply to any Tax imposed by reason of the holder (or the beneficial owne

#### Where re-ranking does not help

In [25]:
# a cross-encoder can only re-order what the bi-encoder retrieved: if the gold chunk is
# not in the top 10, no amount of re-ranking will recover it
lim = pd.DataFrame([run_case(*case) for case in LIMITATION_QUERIES])
lim[["category", "query", "gold_chunk", "rank_shift", "gold_in_P1_top3", "gold_in_P2_top3"]]

[Recall failure]  gold chunk(s): [215]
QUERY: What was the financial impact of the European Court of Justice decision on Apple?
CE rank chunk  page  bi rank  cosine   CE score  shift
#1      214    43    #2       0.5593   -4.161    #2 -> #1  (+1)  P2/P1
#2      103    17    #10      0.5097   -6.718    #10 -> #2  (+8)  P2
#3      150    25    #4       0.5451   -7.537    #4 -> #3  (+1)  P2
#4      181    35    #8       0.5181   -8.944    #8 -> #4  (+4)  
#5      175    32    #7       0.5202   -9.152    #7 -> #5  (+2)  
#6      102    17    #6       0.5400   -9.171    #6 -> #6  
#7      177    33    #3       0.5502   -9.310    #3 -> #7  (-4)  P1
#8      104    17    #1       0.5655   -9.792    #1 -> #8  (-7)  P1
#9      132    21    #5       0.5429   -10.510   #5 -> #9  (-4)  
#10     279    59    #9       0.5131   -10.583   #9 -> #10  (-1)  

Pipeline 1 top-3: [104, 214, 177]   Pipeline 2 top-3: [214, 103, 150]
promoted into top-3: [103, 150]   dropped from top-3: [104, 177]

ANSWER 1 (b

,category,query,gold_chunk,rank_shift,gold_in_P1_top3,gold_in_P2_top3
0,Recall failure,What was the financial impact of the European Court of Justice decision on A...,215,not in top 10 -> not in top 10,False,False
1,Recall failure,Which executive is departing Apple?,264,not in top 10 -> not in top 10,False,False
2,Re-ranker demotes gold,Which taxes are excluded from Additional Amounts?,313,#6 -> #7,False,False


### 4. Latency Profiling

In [26]:
all_runs = pd.concat([df, lim], ignore_index=True)
lat = pd.DataFrame({
    "Pipeline 1 (baseline)": {
        "retrieval (ms)": all_runs.p1_retrieval_ms.mean(),
        "re-rank (ms)": 0.0,
        "retrieval total (ms)": all_runs.p1_retrieval_ms.mean(),
        "LLM generation (s)": all_runs.p1_llm_s.mean()},
    "Pipeline 2 (re-rank)": {
        "retrieval (ms)": all_runs.p2_retrieval_ms.mean(),
        "re-rank (ms)": all_runs.rerank_ms.mean(),
        "retrieval total (ms)": (all_runs.p2_retrieval_ms + all_runs.rerank_ms).mean(),
        "LLM generation (s)": all_runs.p2_llm_s.mean()},
}).round(2)
print(f"mean over {len(all_runs)} queries")
lat

mean over 10 queries


,Pipeline 1 (baseline),Pipeline 2 (re-rank)
retrieval (ms),9.08,8.46
re-rank (ms),0.00,13.62
retrieval total (ms),9.08,22.07
LLM generation (s),0.77,1.57
